Environment Variables

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install llama-cpp-python

# Pre-Setup

* Mounting Drive
* Setting file path for model and csv files
* Specifying bert model for evaluation



In [9]:
from google.colab import drive
drive.mount('/content/drive')
model_path = "/content/drive/My Drive/Thesis/Models/Model Selection - Phase 1/qwen3-4b-instruct-2507.Q4_K_M.gguf"
inference_csv_file_path = '/content/ilocano_filtered_data.csv'
golden_csv_path = inference_csv_file_path
bert_model="google-bert/bert-base-multilingual-cased"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading of your GGUF model using `Llama.from_pretrained`

In [10]:
from llama_cpp import Llama
import os

# Define max_seq_length as it is a dependency for Llama initialization
# This value defines the context window size for the model.
max_seq_length = 2048 # Common context window size for Llama models

# Ensure the model path is correct
if not os.path.exists(model_path):
    print(f"Error: Model file not found at {model_path}. Please ensure Google Drive is mounted and the path is correct.")
else:
    print(f"Loading model from: {model_path}")
    # Initialize the Llama model
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1, # Uncomment to use GPU if available, -1 to offload all layers to GPU
        n_ctx=max_seq_length, # Context window size
        verbose=False # Suppress llama_cpp verbose output
    )
    print("Model loaded successfully!")

Loading model from: /content/drive/My Drive/Thesis/Models/Model Selection - Phase 1/qwen3-4b-instruct-2507.Q4_K_M.gguf


llama_context: n_ctx_per_seq (2048) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


Model loaded successfully!


Inference and saving to all outputs to CSV


In [11]:
import csv
import pandas as pd
import os

def run_inference_and_save(user_prompts, system_prompts=None, output_csv_path='inference_results.csv'):
    """
    Runs inference with the loaded Llama model for a list of user prompts and saves the results to a CSV file.
    Optionally, it can include system prompts if provided.

    Args:
        user_prompts (list): A list of user input prompts (strings).
        system_prompts (list, optional): A list of corresponding system prompts (strings).
                                          Must be the same length as user_prompts if provided.
        output_csv_path (str): The path to the CSV file where results will be saved.
    """
    if not hasattr(globals().get('llm'), 'create_chat_completion'):
        print("Error: LLM model not loaded. Please run the model loading cells first.")
        return

    if system_prompts is not None and len(user_prompts) != len(system_prompts):
        print("Error: Length of user_prompts and system_prompts must be the same.")
        return

    # To ensure a fresh run, remove the existing output file if it exists
    if os.path.exists(output_csv_path):
        os.remove(output_csv_path)
        print(f"Removed existing '{output_csv_path}' for a fresh run.")

    print(f"Starting inference for {len(user_prompts)} prompts...")

    try:
        with open(output_csv_path, 'w', newline='', encoding='utf-8') as f: # Use 'w' to always start fresh
            writer = csv.writer(f)
            # Determine header based on whether system prompts are used
            if system_prompts is not None:
                writer.writerow(['System Prompt', 'prompt', 'response'])
            else:
                writer.writerow(['User Prompt', 'Model Response'])

            for i, user_prompt in enumerate(user_prompts):
                print(f"\n\n\nProcessing prompt {i+1}/{len(user_prompts)}: {user_prompt[:300]}...") # Log progress

                messages = []
                current_system_prompt = None
                if system_prompts is not None:
                    current_system_prompt = system_prompts[i]
                    if current_system_prompt:
                        messages.append({"role": "system", "content": current_system_prompt})
                        print(f"  Utilizing System Prompt: {current_system_prompt[:300]}...")

                messages.append({"role": "user", "content": user_prompt})

                # Perform inference
                output = llm.create_chat_completion(
                    messages=messages,
                    max_tokens=256,
                    temperature=0.7,
                    stream=False,
                    stop=["<|eot_id|>", "<|start_header_id|>user<|end_header_id>", "user", "assistant", "átkyuser"]
                )

                # Extract the model's response
                model_response = output['choices'][0]['message']['content'].strip()
                print(f"  Model Response: {model_response[:300]}...") # Log response snippet

                # Write row based on whether system prompts are used
                if system_prompts is not None:
                    writer.writerow([current_system_prompt, user_prompt, model_response])
                else:
                    writer.writerow([user_prompt, model_response])

        print(f"All inference results saved to {output_csv_path}")

    except Exception as e:
        print(f"An error occurred during inference or saving: {e}")

# --- Example usage: Load prompts from cebuano_filtered_data.csv ----
if not os.path.exists(inference_csv_file_path):
    print(f"Error: Input CSV file not found at {inference_csv_file_path}.")
else:
    prompts_df = pd.read_csv(inference_csv_file_path)

    # Extract user prompts from the 'prompt' column
    user_prompts_for_inference = prompts_df['prompt'].tolist()

    # Extract system prompts from the 'system' column if it exists
    system_prompts_for_inference = None
    if 'system' in prompts_df.columns:
        system_prompts_for_inference = prompts_df['system'].tolist()

    print(f"Loaded {len(user_prompts_for_inference)} user prompts from {inference_csv_file_path}.")
    if system_prompts_for_inference:
        print(f"Loaded {len(system_prompts_for_inference)} system prompts.")

    if not user_prompts_for_inference:
        print(f"No user prompts found in the 'prompt' column of {inference_csv_file_path}.")
    else:
        run_inference_and_save(
            user_prompts_for_inference,
            system_prompts=system_prompts_for_inference,
            output_csv_path='inference_output.csv'
        )

Loaded 301 user prompts from /content/ilocano_filtered_data.csv.
Loaded 301 system prompts.
Removed existing 'inference_output.csv' for a fresh run.
Starting inference for 301 prompts...



Processing prompt 1/301: damag: Mangaramid iti ababa a sarita a pakairamanan ti ilelennek ti init ken sekreto....
  Utilizing System Prompt: Sika ket maysa a makatulong nga assistant....


KeyboardInterrupt: 

### Computing BERTScore

To compute BERTScore, we need the model's generated responses (candidates) and the corresponding human-written reference answers (golden answers).

In [ ]:
%%capture
!pip install bert_score

### Assigning Lower Weights for Common Words with IDF

BERTScore can incorporate Inverse Document Frequency (IDF) weighting to give less importance to very common words (like "the", "is", "a"), and more importance to rare words. This is often beneficial because highly frequent words carry less semantic information.

By setting `idf=True` in the `score` function, BERTScore will compute IDF values for each token and use these as weights when calculating the similarity scores. This means tokens that appear frequently across the reference texts will have lower weights, effectively down-weighting their contribution to the overall score.

In [ ]:
import pandas as pd
from bert_score import score
import os
import re # Added import for regular expressions

# Load model responses
output_csv_path = 'inference_output.csv'
if not os.path.exists(output_csv_path):
    print(f"Error: {output_csv_path} not found. Please ensure inference has been run and the file exists.")
else:
    model_responses_df = pd.read_csv(output_csv_path)
    print(f"Loaded model responses from {output_csv_path}:")
    display(model_responses_df.head())

# Load golden answers
golden_answers_df = pd.read_csv(golden_csv_path)
print(f"Loaded golden answers from {golden_csv_path}:")
display(golden_answers_df.head())

# Merge the two dataframes on the 'prompt' column
if 'prompt' in model_responses_df.columns and 'prompt' in golden_answers_df.columns:
    merged_df = pd.merge(model_responses_df, golden_answers_df, on='prompt', how='inner')
    print("Merged DataFrame:")
    display(merged_df.head())

    if not merged_df.empty:
        # Prepare candidates and references
        # Ensure all entries are strings to avoid AttributeError on .lower()
        candidates = merged_df['response_x'].astype(str).tolist() # Model responses are in 'response_x'
        references = merged_df['response_y'].astype(str).tolist() # Golden answers are in 'response_y'

        # --- Custom Word List Processing ---
        # Define your custom word list (e.g., stop words or words to deemphasize)
        #custom_word_list = ['a', 'the', 'is', 'of', 'in'] # Example list
        custom_word_list = ['<tool_call>', '</tool_call>', '</think>', '<think>'] # Set to none

        # Function to remove words from a text and normalize whitespace
        def remove_words_from_text(text, word_list):
            # Ensure text is treated as a string, even if it was originally 'nan'
            text = str(text)
            # Normalize whitespace: replace any sequence of whitespace characters (including newlines)
            # with a single space and strip leading/trailing whitespace.
            text = re.sub(r'\s+', ' ', text).strip()
            words = text.lower().split() # Convert to lowercase and split into words
            filtered_words = [word for word in words if word not in word_list]
            return ' '.join(filtered_words)

        print("\nPreprocessing candidates and references with custom word list...")
        preprocessed_candidates = [remove_words_from_text(c, custom_word_list) for c in candidates]
        preprocessed_references = [remove_words_from_text(r, custom_word_list) for r in references]

        # Optional: Display a few preprocessed examples
        print(f"Original Candidate (first): {candidates[0]}")
        print(f"Preprocessed Candidate (first): {preprocessed_candidates[0]}")
        print(f"Original Reference (first): {references[0]}")
        print(f"Preprocessed Reference (first): {preprocessed_references[0]}")

        print("\nComputing BERTScore with preprocessed texts...")
        # Compute BERTScore using the preprocessed texts
        # You can specify a different model type. For example, for mBERT, use "bert-base-multilingual-cased".
        # If you have a custom model like "cebuano_based_cased_bert", ensure it's a valid Hugging Face model identifier or installed.
        # idf=True can still be used here to further down-weight other common words in the *remaining* text
        P, R, F1 = score(preprocessed_candidates, preprocessed_references, model_type=bert_model, num_layers=12, idf=True, verbose=True, use_fast_tokenizer=False)

        # Add BERTScore to the merged DataFrame
        merged_df['BERTScore_P_preprocessed'] = P.numpy()
        merged_df['BERTScore_R_preprocessed'] = R.numpy()
        merged_df['BERTScore_F1_preprocessed'] = F1.numpy()

        print("\nBERTScore Results per sample (with custom word list preprocessing):")
        display(merged_df[['prompt', 'response_x', 'response_y', 'BERTScore_P_preprocessed', 'BERTScore_R_preprocessed', 'BERTScore_F1_preprocessed']])

        print(f"\nAverage BERTScore F1 (with custom word list preprocessing): {F1.mean():.4f}")
        print(f"Average BERTScore Precision (with custom word list preprocessing): {P.mean():.4f}")
        print(f"Average BERTScore Recall (with custom word list preprocessing): {R.mean():.4f}")
    else:
        print("No matching prompts found between 'inference_output.csv' and 'golden_answers.csv'. BERTScore cannot be computed.")
else:
    print("Error: 'prompt' column not found in one or both CSV files. Please check the column names.")

# Task
Save the `merged_df` DataFrame, including the newly calculated BERTScore metrics, to a CSV file named `inference_output.csv`, then verify that the file now contains the BERTScore outputs.

## Save BERTScore Outputs to CSV

### Subtask:
Save the `merged_df` (which contains the original prompts, model responses, golden answers, and calculated BERTScore metrics) to `inference_output.csv`. This will update the file to include all BERTScore outputs.


**Reasoning**:
To save the `merged_df` DataFrame, which now contains the BERTScore metrics, to a CSV file named `inference_output.csv` without including the DataFrame index.



In [12]:
merged_df.to_csv('inference_output.csv', index=False)
print("Updated 'inference_output.csv' with BERTScore outputs.")

Updated 'inference_output.csv' with BERTScore outputs.


**Reasoning**:
To verify that the `inference_output.csv` file has been updated correctly with the BERTScore outputs, I will load the CSV back into a DataFrame and display the first few rows.



In [ ]:
import pandas as pd

# Load the updated CSV file
updated_df = pd.read_csv('inference_output.csv')

# Display the first few rows to verify the BERTScore columns
print("Verifying updated 'inference_output.csv' with BERTScore outputs:")
display(updated_df.head())

## Final Task

### Subtask:
Confirm that the `inference_output.csv` file now contains the BERTScore outputs.


## Summary:

### Q&A
Yes, the `inference_output.csv` file now contains the BERTScore outputs.

### Data Analysis Key Findings
*   The `merged_df` DataFrame, which includes original prompts, model responses, golden answers, and BERTScore metrics, was successfully saved to `inference_output.csv`.
*   Verification confirmed that the `inference_output.csv` file now contains the `BERTScore_P_preprocessed`, `BERTScore_R_preprocessed`, and `BERTScore_F1_preprocessed` columns, indicating the successful integration and saving of BERTScore outputs.

### Insights or Next Steps
*   The updated `inference_output.csv` can now be used for further analysis, reporting, or as input for downstream processes that require both the original data and the BERTScore evaluation metrics.
